# Item #12: field-panel-grid figures, generated by the ORIGINAL study scripts themselves

**Omar's own explicit direction**, after flagging a real risk in an earlier version of this item: a separate script that
reconstructs a trained model's own hyperparameters independently could load the wrong ones for some checkpoint. Fixed not by
double-checking that reconstruction, but by removing the reconstruction entirely — a new `--save_sample_plot` flag was
added DIRECTLY to `resolution_invariance_zeroshot.py`'s `cmd_eval` and `ood_progressive.py`'s `main`, at the exact point each
already computes a prediction. This cell re-invokes each study's OWN real original command (the same one that already produced
Tables 12/19/25/26's own published numbers) with that one flag added — nothing about the model-loading or prediction code is
re-derived, so there is nothing new to get wrong.

**Verified additive and non-destructive** before trusting it: run twice locally against a small test checkpoint, once fresh
and once resumed from a cached `out_json` — both produced the image, and the numeric output was byte-identical to the same
command without the flag (see PROJECT_STATUS.md item #12).

**Why re-running the whole project is fast**: every command below already ran to completion before this session — each
script's own resumability logic detects that and skips recomputing it. The only new work is one extra sample per
resolution/shift level, purely for the plot.

* **A — FEM convergence (Table 6a/20), B1 Neo-Hookean.** No model here at all (the checkpoint is a raw converged solution
  vector) — this section never had the risk described above.
* **B — Zero-shot resolution (Table 12), B1 Neo-Hookean.** Real eval command from this checkpoint's own `run_manifest.json`.
* **C — OOD shift (Tables 19/25), all 6 geometry x material cases.** Real checkpoints/commands from `cell_ood_progressive.py`
  and `cell_ood_progressive_remaining.py` (already committed).
* **D — DD-NO coarse-vs-fine (Table 26), B1 Neo-Hookean.** Same eval script as B, pointed at Table 26's own two checkpoints.

**Each section is independent and wrapped in its own try/except** — if one checkpoint path does not match your actual
Drive layout, that section prints its real error and the others still run. Fix the path constant named in that section
and re-run just this cell.

* No GPU strictly required for A; a GPU speeds up B/C/D's forward passes and C's small (N=21) FEM solves, but nothing
  here is large-scale.
* Saves every figure to `/content/drive/MyDrive/pfem_run/figures/` — fetch them from Drive afterward.


In [ ]:
# =====================================================================
#  CELL -- Item #12: field-panel-grid figures for every study with a
#  saved checkpoint, generated by RE-RUNNING EACH STUDY'S OWN REAL
#  ORIGINAL COMMAND (exactly what already produced Tables 12/19/25/26's
#  own published numbers) with one new opt-in flag, --save_sample_plot,
#  added directly to that same script.
#
#  WHY THIS WAY, NOT A SEPARATE SCRIPT (Omar's own explicit direction):
#  a separate script that reconstructs the model/args itself risks using
#  the wrong hyperparameters for a given checkpoint, which either fails
#  loudly (safe) or, worse, could silently mismatch in some future case.
#  Adding the plot-saving code directly inside the ORIGINAL, already-
#  validated function -- resolution_invariance_zeroshot.py's cmd_eval,
#  ood_progressive.py's main -- and re-invoking it with the real
#  original command means there is nothing new to get wrong: the exact
#  same model-loading, mesh-building and prediction code that produced
#  the published numbers also produces the picture.
#
#  --save_sample_plot is opt-in and purely additive: verified locally
#  (both a fresh run and a resumed/cached run) that adding it reproduces
#  byte-identical numeric output to the same command without it -- see
#  PROJECT_STATUS.md item #12 for the two side-by-side test runs.
#
#  WHY THIS IS FAST even though it "re-runs the whole project": every
#  one of these commands is the SAME command already run to completion
#  before, and each script's own resumability logic detects that
#  in-json/on-Drive and skips recomputing it -- the only genuinely new
#  work is capturing one extra sample per resolution/shift level purely
#  for the plot (a cheap forward pass, or for OOD a cheap N=21 FEM solve
#  since no --cache_dir was used in the original OOD runs), not
#  reproducing the actual training or the full evaluation sweep.
#
#  Each section is independent and wrapped in try/except -- if one
#  checkpoint path below does not match your actual Drive layout, that
#  section prints its real error and the others still run.
# =====================================================================
import os, sys, subprocess

from google.colab import drive
drive.mount('/content/drive')
from IPython.display import Image, display

REPO = '/content/OMAR'
def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if _mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.'):
        del sys.modules[_mod_name]

R = '/content/drive/MyDrive/pfem_run'
CKPT_DIR = '/content/drive/MyDrive/pfem_ckpt'
FIG = f'{R}/figures'
os.makedirs(FIG, exist_ok=True)

failures = []

# --- Section A: FEM convergence field grid (Table 6a/20), B1 Neo-Hookean.
# Unchanged from the first version of this item -- no model to reconstruct
# here at all (the checkpoint is the raw converged solution vector, not a
# trained network), so this section never had the risk the others below
# are built to eliminate.
print('\n=== A: FEM convergence field grid ===')
try:
    from omar_pfem.field_snapshot_grid import make_resolution_grid_figure
    make_resolution_grid_figure(
        [51, 101, 201, 401, 701, 1001, 1401], CKPT_DIR,
        f'{FIG}/fig_B1_resolution_grid.png')
except Exception as e:
    print('SECTION A FAILED:', e); failures.append(('A', e))

# --- Section B: zero-shot resolution (Table 12), B1 Neo-Hookean.
# Exact real command from this checkpoint's own run_manifest.json.
print('\n=== B: zero-shot resolution field grid (Table 12) ===')
try:
    run([sys.executable, '-u', '-m', 'omar_pfem.resolution_invariance_zeroshot', 'eval',
         '--geometry', 'B1', '--material', 'neo_hookean',
         '--checkpoint', f'{R}/zeroshot_B1_neo_hookean/model_best.pt',
         '--test_resolutions', '13,17,25,29,37,41,49', '--fine_N', '101',
         '--n_eval_samples', '20',
         '--out_json', f'{R}/zeroshot_B1_neo_hookean/zeroshot_eval_coarse_and_fine.json',
         '--save_sample_plot', f'{FIG}/fig_B1_zeroshot_grid.png'])
except Exception as e:
    print('SECTION B FAILED:', e); failures.append(('B', e))

# --- Section C: OOD shift (Tables 19/25), all 6 geometry x material cases.
# Exact real checkpoints/commands from cell_ood_progressive.py and
# cell_ood_progressive_remaining.py (already committed to this repo).
print('\n=== C: OOD shift field grids (Tables 19/25), 6 cases ===')
OOD_CASES = [
    ('B1', 'neo_hookean', f'{R}/results/B1_neo_hookean/model_best.pt'),
    ('B1', 'mooney_rivlin', f'{R}/results/B1_mooney_rivlin/model_best.pt'),
    ('B1', 'arruda_boyce', f'{R}/results/B1_arruda_boyce/model_best.pt'),
    ('B2', 'neo_hookean', f'{R}/B2_accuracy_search/lossnorm/train/model_best.pt'),
    ('B2', 'mooney_rivlin', f'{R}/B2_accuracy_search_mooney_rivlin/lossnorm/train/model_best.pt'),
    ('B2', 'arruda_boyce', f'{R}/B2_accuracy_search_arruda_boyce/lossnorm/train/model_best.pt'),
]
for geometry, material, ckpt in OOD_CASES:
    try:
        run([sys.executable, '-u', '-m', 'omar_pfem.ood_progressive',
             '--geometry', geometry, '--material', material, '--checkpoint', ckpt,
             '--N', '21', '--shifts', '0,0.5,1.0,1.5,2.0,2.5,3.0',
             '--factors', 'material,loading,both', '--n_samples', '10',
             '--out_json', f'{R}/ood_progressive/ood_progressive_{geometry}_{material}.json',
             '--save_sample_plot', f'{FIG}/fig_{geometry}_{material}_ood_grid.png'])
    except Exception as e:
        print(f'SECTION C ({geometry} x {material}) FAILED:', e)
        failures.append((f'C_{geometry}_{material}', e))

# --- Section D: DD-NO coarse-vs-fine (Table 26), B1 Neo-Hookean.
# Same eval script as Section B, pointed at each of Table 26's own two
# checkpoints -- produces two images (coarse-trained, fine-trained),
# directly comparable side by side.
print('\n=== D: DD-NO coarse-vs-fine field grids (Table 26) ===')
DD_NO_CASES = [
    ('coarse', f'{R}/dd_no_coarse_vs_fine/coarse_N13_800tr_200te_75000steps_train/model_best.pt',
     f'{R}/dd_no_coarse_vs_fine/coarse_N13_800tr_200te_75000steps_zeroshot.json'),
    ('fine', f'{R}/dd_no_coarse_vs_fine/fine_N33_800tr_200te_75000steps_train/model_best.pt',
     f'{R}/dd_no_coarse_vs_fine/fine_N33_800tr_200te_75000steps_zeroshot.json'),
]
for label, ckpt, out_json in DD_NO_CASES:
    try:
        run([sys.executable, '-u', '-m', 'omar_pfem.resolution_invariance_zeroshot', 'eval',
             '--geometry', 'B1', '--material', 'neo_hookean', '--checkpoint', ckpt,
             '--test_resolutions', '13,17,25,29,37,41,49', '--fine_N', '101',
             '--n_eval_samples', '20', '--out_json', out_json,
             '--save_sample_plot', f'{FIG}/fig_B1_dd_no_{label}_grid.png'])
    except Exception as e:
        print(f'SECTION D ({label}) FAILED:', e); failures.append((f'D_{label}', e))

print('\n' + '=' * 70)
print(f'DONE. Figures saved under {FIG}.')
if failures:
    print(f'\n{len(failures)} section(s) failed -- check the real error printed above')
    print('each, fix that section\'s own checkpoint/path constant, and re-run just it:')
    for name, e in failures:
        print(f'  {name}: {e}')
else:
    print('All sections succeeded.')

# --- Section E: show every figure INLINE, right here, at the end of this
# cell/notebook -- so there is no need to go open Drive separately to see
# whether a given figure actually looks right. Reads straight off disk
# (the same files each section above just saved), so this never depends on
# the sections above having succeeded -- whatever exists gets shown,
# whatever is missing is reported by name instead of silently skipped.
print('\n' + '=' * 70)
print('=== E: all figures, inline ===')
EXPECTED_FIGS = [
    ('A: FEM convergence field grid (B1)', f'{FIG}/fig_B1_resolution_grid.png'),
    ('B: zero-shot resolution field grid (Table 12, B1)', f'{FIG}/fig_B1_zeroshot_grid.png'),
] + [
    (f'C: OOD shift field grid ({g} x {m})', f'{FIG}/fig_{g}_{m}_ood_grid.png')
    for g, m, _ in OOD_CASES
] + [
    ('D: DD-NO coarse-trained field grid (B1)', f'{FIG}/fig_B1_dd_no_coarse_grid.png'),
    ('D: DD-NO fine-trained field grid (B1)', f'{FIG}/fig_B1_dd_no_fine_grid.png'),
]
for title, path in EXPECTED_FIGS:
    print(f'\n--- {title} ---')
    if os.path.isfile(path):
        display(Image(filename=path))
    else:
        print(f'  (missing -- {path} was not produced, see that section\'s error above)')
